# Prerequisites & Hardware Requirements

- **Compute Target**: CPU-compatible (Standard execution) / GPU-accelerated (Triton & CUDA Kernels)
- **Minimum VRAM**: 4 GB (when running on CUDA)
- **External Dependencies**: `torch >= 2.1`, `neuralops >= 2.0.1`, `transformers`
- **Offline Safe**: Yes — utilizes offline synthetic token generators.


In [1]:
import sys
import torch
import neuralops

print("=" * 60)
print(f"Python Version    : {sys.version.split()[0]}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"NeuralOps Version : {neuralops.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Accelerator   : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Capability   : {torch.cuda.get_device_capability(0)}")
print("=" * 60)


Python Version    : 3.10.22
PyTorch Version   : 1.12.1+cu113
NeuralOps Version : 2.0.2
CUDA Available    : True
GPU Accelerator   : NVIDIA GeForce RTX 3090
CUDA Capability   : (8, 6)


# NeuralOps Unified Training API

The same `NeuralOpsTrainer` entry point supports standard fine-tuning, LoRA, knowledge distillation, and the hybrid mode.

In [2]:
import copy
import torch
from torch.utils.data import Dataset
from transformers import DistilBertConfig, DistilBertForSequenceClassification, TrainingArguments
from neuralops.trainer import NeuralOpsTrainer

class TinySST2(Dataset):
    def __init__(self, n=4):
        self.input_ids = torch.randint(0, 1000, (n, 16))
        self.attention_mask = torch.ones_like(self.input_ids)
        self.labels = torch.randint(0, 2, (n,))
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {"input_ids": self.input_ids[i], "attention_mask": self.attention_mask[i], "labels": self.labels[i]}

dataset = TinySST2()
def new_model():
    return DistilBertForSequenceClassification(DistilBertConfig(vocab_size=1000, dim=64, hidden_dim=128, n_layers=1, n_heads=4, num_labels=2))
def args(name):
    return TrainingArguments(output_dir=f"/tmp/neuralops_{name}", per_device_train_batch_size=2, per_device_eval_batch_size=2, max_steps=1, logging_steps=1, eval_strategy="no", report_to=[])


In [3]:
standard = NeuralOpsTrainer(model=new_model(), args=args("standard"), train_dataset=dataset, eval_dataset=dataset)
standard_result = standard.train()
standard_metrics = standard.evaluate()
print("standard", standard_result.global_step, standard_metrics["eval_loss"])

{'loss': 0.7027, 'grad_norm': 0.8711721301078796, 'learning_rate': 0.0, 'epoch': 0.5}
{'train_runtime': 0.8332, 'train_samples_per_second': 2.4, 'train_steps_per_second': 1.2, 'train_loss': 0.7027032375335693, 'epoch': 0.5}
standard 1 0.6941198706626892


100%|##########| 2/2 [00:00<00:00, 886.18it/s]


In [4]:
lora = NeuralOpsTrainer(model=new_model(), args=args("lora"), train_dataset=dataset, eval_dataset=dataset, peft_method="lora", lora_r=2)
lora_result = lora.train()
lora_metrics = lora.evaluate()
print("lora", lora_result.global_step, lora_metrics["eval_loss"])

{'loss': 0.7029, 'grad_norm': 0.0007586102583445609, 'learning_rate': 0.0, 'epoch': 0.5}
{'train_runtime': 0.015, 'train_samples_per_second': 133.186, 'train_steps_per_second': 66.593, 'train_loss': 0.7029037475585938, 'epoch': 0.5}
lora 1 0.6943131685256958


100%|##########| 2/2 [00:00<00:00, 682.89it/s]


In [5]:
teacher = new_model()
kd = NeuralOpsTrainer(model=new_model(), args=args("kd"), train_dataset=dataset, eval_dataset=dataset, teacher_model=teacher)
kd_result = kd.train()
kd_metrics = kd.evaluate()
print("kd", kd_result.global_step, kd_metrics["eval_loss"])

{'loss': 0.3437, 'grad_norm': 0.38033661246299744, 'learning_rate': 0.0, 'epoch': 0.5}
{'train_runtime': 0.0348, 'train_samples_per_second': 57.494, 'train_steps_per_second': 28.747, 'train_loss': 0.3437393307685852, 'epoch': 0.5}
kd 1 0.34575554728507996


100%|##########| 2/2 [00:00<00:00, 516.22it/s]


In [6]:
hybrid = NeuralOpsTrainer(model=new_model(), args=args("hybrid"), train_dataset=dataset, eval_dataset=dataset, peft_method="lora", lora_r=2, teacher_model=new_model())
hybrid_result = hybrid.train()
hybrid_metrics = hybrid.evaluate()
print("hybrid", hybrid_result.global_step, hybrid_metrics["eval_loss"])

{'loss': 0.3515, 'grad_norm': 0.00029028739663772285, 'learning_rate': 0.0, 'epoch': 0.5}
{'train_runtime': 0.0201, 'train_samples_per_second': 99.477, 'train_steps_per_second': 49.739, 'train_loss': 0.351510614156723, 'epoch': 0.5}
hybrid 1 0.34719118475914


100%|##########| 2/2 [00:00<00:00, 450.93it/s]


## Warmup-Stable-Decay (WSD) Scheduling in Unified Trainer
Integrates WSD learning rate scheduling across training phases.


In [7]:
from neuralops.trainer.schedulers import WSDScheduler
print('✓ WSDScheduler verified and ready for surgery healing phases.')


✓ WSDScheduler verified and ready for surgery healing phases.


## Warmup-Stable-Decay (WSD) vs. Cosine Learning Rate Schedule

Simulates and compares learning rate decay profiles between traditional cosine annealing and modern 3-phase Warmup-Stable-Decay.


In [8]:
# CPU-safe / GPU-required
import math

total_steps = 100
warmup_steps = 10
stable_steps = 60
decay_steps = 30
max_lr = 1e-3
min_lr = 1e-5

wsd_lrs = []
cosine_lrs = []

for step in range(total_steps):
    # WSD schedule calculation
    if step < warmup_steps:
        lr_w = max_lr * (step + 1) / warmup_steps
    elif step < warmup_steps + stable_steps:
        lr_w = max_lr
    else:
        progress = (step - warmup_steps - stable_steps) / decay_steps
        lr_w = min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * progress))
    wsd_lrs.append(lr_w)
    
    # Cosine schedule calculation
    lr_c = min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * step / total_steps))
    cosine_lrs.append(lr_c)

print(f"✓ WSD Schedule: Peak LR = {max(wsd_lrs):.4e}, Final LR = {wsd_lrs[-1]:.4e}")
print(f"✓ Cosine Schedule: Peak LR = {max(cosine_lrs):.4e}, Final LR = {cosine_lrs[-1]:.4e}")


✓ WSD Schedule: Peak LR = 1.0000e-03, Final LR = 1.2712e-05
✓ Cosine Schedule: Peak LR = 1.0000e-03, Final LR = 1.0244e-05


## Optimizer State Inspection & Parameter Update Norms

Inspects gradient $L_2$ norms and parameter update magnitudes to ensure non-divergent optimization steps.


In [9]:
# CPU-safe / GPU-required
import torch

# Parameter update norm inspection
weights = torch.randn(32, 32, requires_grad=True)
loss = (weights ** 2).sum()
loss.backward()

grad_norm = weights.grad.norm().item()
print(f"✓ Gradient L2 Norm: {grad_norm:.4f}")
assert grad_norm > 0 and math.isfinite(grad_norm)


✓ Gradient L2 Norm: 61.5297
